In [16]:
import os
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [17]:
pip install monai

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os, random, numpy as np, torch
from monai.utils import set_determinism

SEED = 42

# Global seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# MONAI determinism
set_determinism(seed=SEED)

# Optional: fix environment hash seed


os.environ["PYTHONHASHSEED"] = str(SEED)


In [19]:
import os
import glob

dataset_dir = "/kaggle/input/hvsmrdata1/Mild"

# List files with exact suffixes (no .gz here)
image_files_mild = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped.nii")))
label_files_mild = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped_seg.nii")))

print(f"Found {len(image_files_mild)} images")
print(f"Found {len(label_files_mild)} labels")

# Sanity check first 3 files
print("Sample image files:", image_files_mild[:3])
print("Sample label files:", label_files_mild[:3])

Found 36 images
Found 36 labels
Sample image files: ['/kaggle/input/hvsmrdata1/Mild/pat10_aug1_cropped.nii', '/kaggle/input/hvsmrdata1/Mild/pat10_aug2_cropped.nii', '/kaggle/input/hvsmrdata1/Mild/pat10_cropped.nii']
Sample label files: ['/kaggle/input/hvsmrdata1/Mild/pat10_aug1_cropped_seg.nii', '/kaggle/input/hvsmrdata1/Mild/pat10_aug2_cropped_seg.nii', '/kaggle/input/hvsmrdata1/Mild/pat10_cropped_seg.nii']


In [20]:
import os
import glob

dataset_dir = "/kaggle/input/hvsmrdata1/Moderate"

# List files with exact suffixes (no .gz here)
image_files_moder = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped.nii")))
label_files_moder = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped_seg.nii")))

print(f"Found {len(image_files_moder)} images")
print(f"Found {len(label_files_moder)} labels")

# Sanity check first 3 files
print("Sample image files:", image_files_moder[:3])
print("Sample label files:", label_files_moder[:3])

Found 36 images
Found 36 labels
Sample image files: ['/kaggle/input/hvsmrdata1/Moderate/pat0_aug1_cropped.nii', '/kaggle/input/hvsmrdata1/Moderate/pat0_aug2_cropped.nii', '/kaggle/input/hvsmrdata1/Moderate/pat0_aug3_cropped.nii']
Sample label files: ['/kaggle/input/hvsmrdata1/Moderate/pat0_aug1_cropped_seg.nii', '/kaggle/input/hvsmrdata1/Moderate/pat0_aug2_cropped_seg.nii', '/kaggle/input/hvsmrdata1/Moderate/pat0_aug3_cropped_seg.nii']


In [21]:
import os
import glob

dataset_dir = "/kaggle/input/hvsmrdata1/Severe"

# List files with exact suffixes (no .gz here)
image_files_sever = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped.nii")))
label_files_sever = sorted(glob.glob(os.path.join(dataset_dir, "*_cropped_seg.nii")))

print(f"Found {len(image_files_sever)} images")
print(f"Found {len(label_files_sever)} labels")

# Sanity check first 3 files
print("Sample image files:", image_files_sever[:3])
print("Sample label files:", label_files_sever[:3])

Found 36 images
Found 36 labels
Sample image files: ['/kaggle/input/hvsmrdata1/Severe/pat16_cropped.nii', '/kaggle/input/hvsmrdata1/Severe/pat20_cropped.nii', '/kaggle/input/hvsmrdata1/Severe/pat21_cropped.nii']
Sample label files: ['/kaggle/input/hvsmrdata1/Severe/pat16_cropped_seg.nii', '/kaggle/input/hvsmrdata1/Severe/pat20_cropped_seg.nii', '/kaggle/input/hvsmrdata1/Severe/pat21_cropped_seg.nii']


In [22]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd,
    ResizeD, CropForegroundd, ToTensord, NormalizeIntensityd,
    # --- Recommended Augmentations ---
    RandFlipd,          # 1. Random Flipping
    RandAffined,        # 2. Random Rotation/Zoom/Shear
    Rand3DElasticd,     # 3. Non-linear Deformation (Highly effective)
    RandGaussianNoised, # 4. Random Noise
    RandAdjustContrastd # 5. Random Contrast/Brightness
)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys=["image"]),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    ResizeD(keys=["image"], spatial_size=(96, 96, 96), mode="trilinear"),
    ResizeD(keys=["label"], spatial_size=(96, 96, 96), mode="nearest"),

    # ----- Strong Spatial Augmentations -----
    RandFlipd(keys=["image", "label"], spatial_axis=[0, 1, 2], prob=0.5),

    RandAffined(
        keys=["image", "label"],
        mode=("trilinear", "nearest"),
        prob=0.3,
        rotate_range=(0.1, 0.1, 0.1),      # larger rotations
        scale_range=(0.15, 0.15, 0.15),    # stronger zoom in/out
        shear_range=(0.05, 0.05, 0.05),    # introduce shear
        translate_range=(10, 10, 10),      # random translations
        padding_mode="zeros"
    ),

    Rand3DElasticd(
        keys=["image", "label"],
        sigma_range=(6, 8),
        magnitude_range=(80, 120),         # more aggressive deformation
        prob=0.2,
        rotate_range=(0.05, 0.05, 0.05),
        mode=("trilinear", "nearest")
    ),

    # ----- Strong Intensity Augmentations -----
    RandGaussianNoised(keys="image", prob=0.3, std=0.01),     # more noise
    RandAdjustContrastd(keys="image", prob=0.3, gamma=(0.7, 1.3)),  # bigger contrast changes

    ToTensord(keys=["image", "label"]),
])




val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys=["image"]),   # first scale intensities
    CropForegroundd(keys=["image", "label"], source_key="image"),
    ResizeD(keys=["image"], spatial_size=(96, 96, 96), mode="trilinear"),
    ResizeD(keys=["label"], spatial_size=(96, 96, 96), mode="nearest"),
    ToTensord(keys=["image", "label"]),
])

In [23]:
data_mild = [{"image": img, "label": lbl} for img, lbl in zip(image_files_mild, label_files_mild)]
data_moder = [{"image": img, "label": lbl} for img, lbl in zip(image_files_moder, label_files_moder)]
data_sever = [{"image": img, "label": lbl} for img, lbl in zip(image_files_sever, label_files_sever)]

In [24]:
def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

In [25]:
def extract_slices_tensor(vol, mode, image_size=96):
    # vol: [D,H,W] float32 tensor (already normalized)
    if mode == 0:  # axial
        slices = vol
    elif mode == 1:  # coronal
        slices = vol.permute(1,0,2)   # [H,D,W] -> slices along H
    elif mode == 2:  # sagittal
        slices = vol.permute(2,0,1)   # [W,D,H]
    else:
        raise ValueError("mode must be 0,1,2")

    # slices = [S,H,W] → reshape → interpolate in one go
    slices = slices.unsqueeze(1)  # [S,1,H,W]

    slices = F.interpolate(
        slices, size=(image_size, image_size),
        mode='bilinear', align_corners=False
    )
    return slices  # [S,1,H,W]


In [26]:
def extract_slices_tensor_mask(vol, mode, image_size=96):
    # vol: [D,H,W] float32 tensor (already normalized)
    if mode == 0:  # axial
        slices = vol
    elif mode == 1:  # coronal
        slices = vol.permute(1,0,2)   # [H,D,W] -> slices along H
    elif mode == 2:  # sagittal
        slices = vol.permute(2,0,1)   # [W,D,H]
    else:
        raise ValueError("mode must be 0,1,2")

    # slices = [S,H,W] → reshape → interpolate in one go
    slices = slices.unsqueeze(1)  # [S,1,H,W]

    slices = F.interpolate(
        slices, size=(image_size, image_size),
        mode='nearest'
    )

    return slices  # [S,1,H,W]


In [27]:
import torch
from torch.utils.data import Dataset 

# NOTE: extract_slices_tensor and extract_slices_tensor_mask are assumed to be external functions.

class SingleVolumeSliceDataset(Dataset):
    def __init__(self, image_batch, mask_batch, mode=0, image_size=96):
        if not torch.is_tensor(image_batch):
            image_batch = torch.as_tensor(image_batch)
        if not torch.is_tensor(mask_batch):
            mask_batch = torch.as_tensor(mask_batch)

        # image_batch: [B,1,D,H,W]
        B = image_batch.shape[0]

        imgs_all = []
        masks_all = []
        pos_all = [] # New list for positional tensors

        for i in range(B):
            vol = image_batch[i,0]  # [D,H,W]
            msk = mask_batch[i,0]   # [D,H,W]
            
            # --- Slice Extraction ---
            vol_slices = extract_slices_tensor(vol, mode, image_size) # [D, 1, H, W]
            msk_slices = extract_slices_tensor_mask(msk.float(), mode, image_size) # [D, 1, H, W]

            num_slices = vol_slices.shape[0] # D
            
            # --- Positional Encoding Calculation ---
            # 1. Create indices [0, 1, 2, ..., D-1]
            indices = torch.arange(num_slices, dtype=torch.float32)
            
            # 2. Normalize indices to [0.0, 1.0]
            depth_denominator = max(1.0, num_slices - 1.0)
            normalized_pos = indices / depth_denominator # [D]
            
            # 3. Reshape to [D, 1] for concatenation later
            pos_tensor = normalized_pos.unsqueeze(1) # [D, 1]
            
            # --- Append Data ---
            masks_all.append(msk_slices.long().squeeze(1)) # [D, H, W] (Squeeze channel for mask)
            imgs_all.append(vol_slices)
            pos_all.append(pos_tensor) # Append the [D, 1] tensor

        self.img2d = torch.cat(imgs_all, dim=0)    # [N, 1, H, W]
        self.mask2d = torch.cat(masks_all, dim=0)  # [N, H, W]
        self.pos2d = torch.cat(pos_all, dim=0)     # [N, 1] <-- New Positional Data

    def __len__(self):
        return self.img2d.shape[0]

    def __getitem__(self, idx):
        # Returns the 2D image slice, its mask, and its normalized position
        return self.img2d[idx], self.mask2d[idx], self.pos2d[idx]
        
    def get_batch(self):
        # Updated to return the positional data as well, useful for a full-batch approach
        return self.img2d, self.mask2d, self.pos2d

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# NOTE: The classes DoubleConv2D, ContextConv, and DoubleConv3D 
# are assumed to be defined as in the earlier conversation.
class DoubleConv2D(nn.Module):
    """(Conv → BN → ReLU) * 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.conv(x)

class DoubleConv3D(nn.Module):
    """(Conv3D → BN → ReLU) * 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.conv(x)

class UNet_2D(nn.Module):
    def __init__(self, depth, in_channels=1, num_classes=9):
        super().__init__()
        
        self.depth = depth
        
        # 3D Path
        self.enc3d1 = DoubleConv3D(in_channels, 16)
        self.w_conv = nn.Conv3d(
            in_channels=16,
            out_channels=16,
            kernel_size=(self.depth, 1, 1),  # Kernel: 1xDepthx1
            stride=(self.depth, 1, 1),       # Stride: 1xDepthx1
            padding=(0, 0, 0)                # No padding needed
        )
        
        # Encoder (Channel counts updated for 1 Positional Channel)
        # Input Channels = (Previous Output) + 1 (Positional) + 16 (3D Context)
        self.enc1 = DoubleConv2D(18, 64)   # 1 + 1 + 16 = 18
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = DoubleConv2D(81, 128)  # 64 + 1 + 16 = 81
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = DoubleConv2D(145, 256) # 128 + 1 + 16 = 145
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = DoubleConv2D(273, 512) # 256 + 1 + 16 = 273
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv2D(512, 1024)

        # Decoder (Standard UNet configuration, channels unchanged)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv2D(1024, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv2D(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv2D(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv2D(128, 64)

        # Final classifier
        self.out_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x_3d, x_2d, pos): # <--- Accepts three inputs
        # Ensure B=1 for x_3d and get spatial dimensions
        B, C, H, W = x_2d.shape
        
        # --- 1. 3D Context Path (p2_1) ---
        p1 = self.enc3d1(x_3d)                              
        p2_1 = self.w_conv(p1).squeeze(2)                   # [1, 16, H, W]
        p2_1 = p2_1.repeat(B, 1, 1, 1)                      # [B, 16, H, W] - Global Context Map
        
        # --- 2. Positional Map (pos_map) ---
        # pos is [B, 1]. Expand it spatially to match H, W
        pos_map = pos.view(-1, 1, 1, 1).repeat(1, 1, H, W)   # [B, 1, H, W]
        
        # --- 3. Encoder Stage 1 (Full Resolution) ---
        # FUSION: x_2d + pos_map + p2_1
        e1_in = torch.cat([x_2d, pos_map, p2_1], dim=1)     # [B, 18, H, W]
        e1 = self.enc1(e1_in)                               # [B, 64, H, W]
        e1_pooled = self.pool1(e1)                          # [B, 64, H/2, W/2]
        
        # Prepare context for next stage (downsample PE and 3D Context)
        pos_map_d1 = self.pool1(pos_map)                    # [B, 1, H/2, W/2]
        p2_1_d1 = self.pool1(p2_1)                          # [B, 16, H/2, W/2]

        # --- 4. Encoder Stage 2 (Half Resolution) ---
        e2_in = torch.cat([e1_pooled, pos_map_d1, p2_1_d1], dim=1) # [B, 81, H/2, W/2]
        e2 = self.enc2(e2_in)                               # [B, 128, H/2, W/2]
        e2_pooled = self.pool2(e2)
        
        # Prepare context for next stage
        pos_map_d2 = self.pool2(pos_map_d1)
        p2_1_d2 = self.pool2(p2_1_d1)
        
        # --- 5. Encoder Stage 3 (Quarter Resolution) ---
        e3_in = torch.cat([e2_pooled, pos_map_d2, p2_1_d2], dim=1) # [B, 145, H/4, W/4]
        e3 = self.enc3(e3_in)                               # [B, 256, H/4, W/4]
        e3_pooled = self.pool3(e3)

        # Prepare context for next stage
        pos_map_d3 = self.pool3(pos_map_d2)
        p2_1_d3 = self.pool3(p2_1_d2)
        
        # --- 6. Encoder Stage 4 (Eighth Resolution) ---
        e4_in = torch.cat([e3_pooled, pos_map_d3, p2_1_d3], dim=1) # [B, 273, H/8, W/8]
        e4 = self.enc4(e4_in)                               # [B, 512, H/8, W/8]
        e4_pooled = self.pool4(e4)

        # --- 7. Bottleneck ---
        b = self.bottleneck(e4_pooled)                      # [B, 1024, H/16, W/16]

        # --- 8. Decoder (Standard UNet structure) ---
        d4 = self.up4(b)                                    
        d4 = self.dec4(torch.cat([d4, e4], dim=1))          
        
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        # Final classifier
        out = self.out_conv(d1)                             # [B, num_classes, H, W]
        return out

In [29]:
import torch
import torch.nn.functional as F
from monai.metrics import DiceMetric
from tqdm.notebook import tqdm



def evaluate_model_2d(model, loader, loss_function=None, mode=1, image_size=96):
    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    steps = 0
    dice_metric_3d = DiceMetric(reduction="mean", num_classes=9, include_background=False)
    with torch.no_grad():
        progress_bar = tqdm(loader, desc="3D Eval (2D slices)", leave=False)

        for batch_data in progress_bar:
            steps += 1

            images = batch_data["image"]  # shape: (1,1,H,W,D)
            labels = batch_data["label"]  # shape: (1,1,H,W,D)

            # ---- SLICE VOLUME INTO 2D ----
            slicer = SingleVolumeSliceDataset(images, labels, mode=mode, image_size=image_size)
            img2d, mask2d, pos2d = slicer.get_batch()  # [S,1,H,W]

            img2d = img2d.to(device, non_blocking=True).float()
            mask2d = mask2d.to(device, non_blocking=True).long()
            pos2d = pos2d.to(device).float()
            images = images.to(device)

            # ---- 2D MODEL INFERENCE ----
            outputs = model(images, img2d, pos2d)
            preds_2d = torch.argmax(outputs, dim=1)  # [S,H,W]
            # ---- LOSS (optional) ----
            if loss_function is not None:
                loss = loss_function(outputs, mask2d.unsqueeze(1))
                total_loss += loss.item()

            # ---- RECONSTRUCT 3D ----
            y_pred = F.one_hot(preds_2d, num_classes=num_classes).permute(0,3,1,2).float()
            y_true = F.one_hot(mask2d.squeeze(1), num_classes=num_classes).permute(0,3,1,2).float()
            dice_metric_3d(y_pred=y_pred, y=y_true)

            avg_loss = total_loss / steps if loss_function else 0
            running_dice = dice_metric_3d.aggregate().item()
            progress_bar.set_postfix({"Loss": f"{avg_loss:.4f}", "Dice": f"{running_dice:.4f}"})

        mean_dice = dice_metric_3d.aggregate().item()
        mean_loss = total_loss / steps if loss_function else None
        dice_metric_3d.reset()

    return mean_loss, mean_dice

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.data import CacheDataset
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm

# =============================
# User-defined parameters
# =============================
device = torch.device("cuda:0")
num_classes = 9
k_folds = 5
max_epochs = 300
val_interval = 1
use_amp = True
SEED = 42
torch.manual_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
use_amp = True

# =============================
# K-Fold splits
# =============================
kf = KFold(n_splits=k_folds, shuffle=True, random_state=SEED)
mild_folds = list(kf.split(data_mild))
moder_folds = list(kf.split(data_moder))
sever_folds = list(kf.split(data_sever))

# =============================
# Containers for metrics
# =============================
fold_train_loss, fold_train_dice = [], []
fold_val_loss_mild, fold_val_loss_moder, fold_val_loss_sever = [], [], []
fold_val_dice_mild, fold_val_dice_moder, fold_val_dice_sever = [], [], []

# =============================
# FOLD LOOP
# =============================
for fold in range(k_folds):
    print(f"\n🚀 Starting Fold {fold+1}/{k_folds}")
    train_dice = mild_dice = moder_dice = sever_dice = 0.0

    # --- Split data ---
    mild_train_idx, mild_val_idx = mild_folds[fold]
    moder_train_idx, moder_val_idx = moder_folds[fold]
    sever_train_idx, sever_val_idx = sever_folds[fold]

    mild_train_list = [data_mild[i] for i in mild_train_idx]
    moder_train_list = [data_moder[i] for i in moder_train_idx]
    sever_train_list = [data_sever[i] for i in sever_train_idx]

    mild_val_list = [data_mild[i] for i in mild_val_idx]
    moder_val_list = [data_moder[i] for i in moder_val_idx]
    sever_val_list = [data_sever[i] for i in sever_val_idx]

    # --- CacheDataset for faster IO ---
    mild_train_ds = CacheDataset(mild_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)
    moder_train_ds = CacheDataset(moder_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)
    sever_train_ds = CacheDataset(sever_train_list, transform=train_transforms, cache_rate=1.0, num_workers=8)

    mild_val_ds = CacheDataset(mild_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)
    moder_val_ds = CacheDataset(moder_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)
    sever_val_ds = CacheDataset(sever_val_list, transform=val_transforms, cache_rate=1.0, num_workers=4)

    # --- Combine training datasets ---
    train_ds = ConcatDataset([mild_train_ds, moder_train_ds, sever_train_ds])

    # --- DataLoaders ---
    train_loader = DataLoader(
        train_ds, batch_size=1, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    mild_loader = DataLoader(
        mild_val_ds, batch_size=1,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    moder_loader = DataLoader(
        moder_val_ds, batch_size=1,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )
    sever_loader = DataLoader(
        sever_val_ds, batch_size=1,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=worker_init_fn, generator=g
    )

    # --- Model, loss, optimizer, scheduler ---
    model = UNet_2D(in_channels=1, num_classes=9, depth=96).to(device)
    loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False, label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.9, patience=5, min_lr=1e-6)
    dice_metric = DiceMetric(reduction="mean", num_classes=num_classes, include_background=False)
    scaler = torch.amp.GradScaler("cuda")

    # --- Trackers ---
    train_loss_values, train_dice_values = [], []
    mild_dice_f, moder_dice_f, sever_dice_f = [], [], []

    # ✅ NEW: Track validation losses
    val_loss_mild_list, val_loss_moder_list, val_loss_sever_list = [], [], []

    # =============================
    # EPOCH LOOP
    # =============================
    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        step = 0
        dice_metric.reset()

        progress_bar = tqdm(train_loader, desc=f"Fold {fold+1} | Epoch {epoch+1}/{max_epochs}", leave=False)
        for batch_data in progress_bar:
            step += 1
            inputs = batch_data["image"]
            labels = batch_data["label"]
            slicer = SingleVolumeSliceDataset(inputs, labels, mode=1, image_size=96)
            img2d, mask2d, pos2d = slicer.get_batch()

            img2d = img2d.to(device).float()
            mask2d = mask2d.unsqueeze(1).to(device).long()
            pos2d=pos2d.to(device).float()
            inputs=inputs.to(device).float()

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(inputs, img2d, pos2d)
                loss = loss_function(outputs, mask2d.long())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            avg_loss = epoch_loss / step
            progress_bar.set_postfix({"Train_Loss": f"{avg_loss:.4f}"})

        epoch_loss /= step
        train_loss_values.append(epoch_loss)

        # --- Training Dice ---
        model.eval()
        dice_metric.reset()
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
            for batch_data in train_loader:
                inputs = batch_data["image"]
                labels = batch_data["label"]
                slicer = SingleVolumeSliceDataset(inputs, labels, mode=1, image_size=96)
                img2d, mask2d, pos2d = slicer.get_batch()
                img2d = img2d.to(device).float()
                mask2d = mask2d.unsqueeze(1).to(device).long()
                pos2d=pos2d.to(device).float()
                inputs=inputs.to(device).float()
                outputs = model(inputs, img2d, pos2d)
                preds = torch.argmax(outputs, dim=1)
                y_pred = F.one_hot(preds, num_classes=num_classes).permute(0,3,1,2).float()
                y_true = F.one_hot(mask2d.squeeze(1), num_classes=num_classes).permute(0,3,1,2).float()
                dice_metric(y_pred=y_pred, y=y_true)
        train_dice = dice_metric.aggregate().item()
        train_dice_values.append(train_dice)
        print(f"Epoch [{epoch+1}/{max_epochs}] | Train Dice: {train_dice:.4f}")

        # --- Validation ---
        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                val_loss_mild, mild_dice = evaluate_model_2d(model, mild_loader, loss_function)
                val_loss_moder, moder_dice = evaluate_model_2d(model, moder_loader, loss_function)
                val_loss_sever, sever_dice = evaluate_model_2d(model, sever_loader, loss_function)

            avg_val_loss = (val_loss_mild + val_loss_moder + val_loss_sever) / 3
            scheduler.step(avg_val_loss)

            # ✅ Store validation losses and dice
            val_loss_mild_list.append(val_loss_mild)
            val_loss_moder_list.append(val_loss_moder)
            val_loss_sever_list.append(val_loss_sever)
            mild_dice_f.append(mild_dice)
            moder_dice_f.append(moder_dice)
            sever_dice_f.append(sever_dice)
            print(f"Validation - Mild: {mild_dice:.4f} | Moder: {moder_dice:.4f} | Sever: {sever_dice:.4f}")

    # --- Store fold metrics ---
    fold_train_loss.append(train_loss_values)
    fold_train_dice.append(train_dice_values)
    fold_val_loss_mild.append(val_loss_mild_list)
    fold_val_loss_moder.append(val_loss_moder_list)
    fold_val_loss_sever.append(val_loss_sever_list)
    fold_val_dice_mild.append(mild_dice_f)
    fold_val_dice_moder.append(moder_dice_f)
    fold_val_dice_sever.append(sever_dice_f)

    # ✅ PRINT FINAL RESULTS AFTER EACH FOLD
    print(f"\n📊 Fold {fold+1} Summary")
    print("=" * 60)
    print(f"Train Loss per epoch: {train_loss_values}")
    print(f"Train Dice per epoch: {train_dice_values}")
    print(f"Validation Loss (Mild): {val_loss_mild_list}")
    print(f"Validation Loss (Moder): {val_loss_moder_list}")
    print(f"Validation Loss (Sever): {val_loss_sever_list}")
    print(f"Validation Dice (Mild): {mild_dice_f}")
    print(f"Validation Dice (Moder): {moder_dice_f}")
    print(f"Validation Dice (Sever): {sever_dice_f}")


In [32]:
print(f"Train Loss per epoch: {train_loss_values}")
print(f"Train Dice per epoch: {train_dice_values}")
print(f"Validation Loss (Mild): {val_loss_mild_list}")
print(f"Validation Loss (Moder): {val_loss_moder_list}")
print(f"Validation Loss (Sever): {val_loss_sever_list}")
print(f"Validation Dice (Mild): {mild_dice_f}")
print(f"Validation Dice (Moder): {moder_dice_f}")
print(f"Validation Dice (Sever): {sever_dice_f}")

Train Loss per epoch: [2.2385587905134474, 1.827360517921902, 1.7712915298484622, 1.7496855826604933, 1.736007687591371, 1.7255542164757138, 1.7024719090688796, 1.696939726670583, 1.69157355598041, 1.673729888030461, 1.6718700627485912, 1.6694427714461373, 1.6590864488056727, 1.646670205252511, 1.6446478849365598, 1.6362464427947998, 1.62694042353403, 1.6324289526258196, 1.6032711352620805, 1.6089417040348053, 1.5956486094565618, 1.595245334364119, 1.5743684839634668, 1.5686721787566231, 1.5546482446647825, 1.5521300335725148, 1.5319943200974238, 1.5454644149258023, 1.5210028730687641, 1.5312936178275518, 1.5069108534426916, 1.5396651114736284, 1.5127231251625788, 1.488039500656582, 1.4801748281433469, 1.489367391381945, 1.4635597552571977, 1.4753253999210538, 1.4584353282338096, 1.4670362259660448, 1.4705526942298526, 1.4546851047447749, 1.4655446438562303, 1.4474014171532221, 1.4399834303628831, 1.4410116374492645, 1.4363768597443898, 1.429149444614138, 1.43725047934623, 1.4154089135

In [39]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np

def visualize_slice_comparison(model, batch_data, num_classes, mode=1, image_size=96, device="cuda"):
    """
    Visualize predictions vs. ground truth for one 3D volume as 2D slices.
    """
    model.eval()
    images = batch_data["image"].to(device)
    labels = batch_data["label"].to(device)

    # --- Get 2D slices ---
    slicer = SingleVolumeSliceDataset(images, labels, mode=mode, image_size=image_size)
    img2d, mask2d, pos2d = slicer.get_batch()

    img2d = img2d.to(device).float()
    mask2d = mask2d.to(device).long()
    pos2d = pos2d.to(device).float()
    images = images.to(device).float()

    # --- Run inference ---
    with torch.no_grad():
        outputs = model(images, img2d, pos2d)
        preds = torch.argmax(outputs, dim=1)  # [S, H, W]

    # --- Pick a random slice to visualize ---
    slice_idx = np.random.randint(0, preds.shape[0])
    img = img2d[slice_idx, 0].cpu().numpy()
    gt = mask2d[slice_idx, 0].cpu().numpy()
    pred = preds[slice_idx].cpu().numpy()

    # --- Plot ---
    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    axs[0].imshow(img, cmap='gray'); axs[0].set_title("Input Slice")
    axs[1].imshow(gt, cmap='tab10'); axs[1].set_title("Ground Truth")
    axs[2].imshow(pred, cmap='tab10'); axs[2].set_title("Prediction")

    # Overlay
    axs[3].imshow(img, cmap='gray')
    axs[3].imshow(pred, cmap='tab10', alpha=0.4)
    axs[3].set_title("Overlay")

    for ax in axs: ax.axis("off")
    plt.tight_layout()
    plt.show()


In [40]:
def reconstruct_3d_volume(model, batch_data, num_classes, mode=1, image_size=96, device="cuda"):
    model.eval()
    images = batch_data["image"].to(device)
    labels = batch_data["label"].to(device)

    slicer = SingleVolumeSliceDataset(images, labels, mode=mode, image_size=image_size)
    img2d, mask2d, pos2d = slicer.get_batch()

    img2d = img2d.to(device).float()
    mask2d = mask2d.to(device).long()
    pos2d = pos2d.to(device).float()
    images = images.to(device).float()

    with torch.no_grad():
        outputs = model(images, img2d, pos2d)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()  # shape [S,H,W]
        masks = mask2d.squeeze(1).cpu().numpy()

    return preds, masks


In [42]:
# Get a single batch from any validation loader
batch = next(iter(mild_loader))   # or moder_loader / sever_loader

# Visualize the model's prediction vs ground truth
visualize_slice_comparison(model, batch, num_classes=9, device=device)

preds_3d, masks_3d = reconstruct_3d_volume(model, batch, num_classes=9)
mid = preds_3d.shape[0] // 2

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(masks_3d[mid], cmap='tab10'); plt.title("GT mid slice")
plt.subplot(1,3,2); plt.imshow(preds_3d[mid], cmap='tab10'); plt.title("Pred mid slice")
plt.subplot(1,3,3); plt.imshow(preds_3d[mid] - masks_3d[mid], cmap='coolwarm'); plt.title("Diff")
plt.show()


RuntimeError: DataLoader worker (pid(s) 954, 955, 956, 957) exited unexpectedly